# T2URL retrieval evaluation

Measures **engine availability, coverage, and overlap** so the `backend` fallback
chain in `ai/t2url.py` is ordered on evidence rather than habit.

Run this before changing a retrieval default, and paste the summary table into
`governance/model_cards/t2url-retrieval.md`.

**This notebook hits live search engines.** It is intentionally small — a few
queries across four engines. Widening it will get you rate-limited, which is
itself one of the things being measured.

In [ ]:
import sys, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # ai/ — so `import t2url` resolves
from t2url import text_to_urls

ENGINES = ["duckduckgo", "yahoo", "startpage", "yandex"]

# Probe queries: one navigational, one informational, one long-tail.
QUERIES = [
    "Cloudera CDP use cases",
    "iceberg table maintenance best practices",
    "how to throttle a metasearch client politely",
]

MAX_RESULTS = 10
COOLDOWN_S = 2  # be a good citizen between engine calls

## 1. Collect results per engine

Each engine is queried on its own so a throttled engine shows up as a zero rather
than being silently masked by the fallback chain.

In [ ]:
results = {}   # (query, engine) -> list[url]
errors = {}    # (query, engine) -> str

for query in QUERIES:
    for engine in ENGINES:
        try:
            urls = text_to_urls(query, max_results=MAX_RESULTS, backend=engine)
            results[(query, engine)] = urls
        except Exception as exc:
            results[(query, engine)] = []
            errors[(query, engine)] = f"{type(exc).__name__}: {exc}"
        time.sleep(COOLDOWN_S)

print(f"{len(results)} engine/query pairs, {len(errors)} errors")
for key, msg in errors.items():
    print("  ", key, "→", msg)

## 2. Availability and yield

**Availability** = fraction of queries that returned anything at all. This is the
number that should drive fallback order — an engine with great results that answers
half the time belongs behind a duller, reliable one.

**Yield** = mean URLs returned, against a ceiling of `MAX_RESULTS`.

In [ ]:
print(f"{'engine':<12} {'availability':>13} {'mean yield':>11}")
print("-" * 38)
for engine in ENGINES:
    counts = [len(results[(q, engine)]) for q in QUERIES]
    availability = sum(1 for c in counts if c) / len(counts)
    print(f"{engine:<12} {availability:>12.0%} {sum(counts) / len(counts):>11.1f}")

## 3. Overlap between engines

Jaccard similarity of returned URL sets, pooled across queries. **Low overlap is
the argument for a longer fallback chain** — it means adding an engine genuinely
widens coverage instead of re-returning what DuckDuckGo already found.

In [ ]:
def jaccard(a, b):
    if not a and not b:
        return float("nan")
    return len(a & b) / len(a | b)

pooled = {e: {u for q in QUERIES for u in results[(q, e)]} for e in ENGINES}

print(f"{'':<12}" + "".join(f"{e[:9]:>11}" for e in ENGINES))
for a in ENGINES:
    row = "".join(f"{jaccard(pooled[a], pooled[b]):>11.2f}" for b in ENGINES)
    print(f"{a:<12}{row}")

union = set().union(*pooled.values()) if pooled else set()
ddg = pooled.get("duckduckgo", set())
if union:
    print(f"\nunique URLs across all engines: {len(union)}")
    print(f"found by duckduckgo alone:      {len(ddg)} ({len(ddg) / len(union):.0%} of union)")

## 4. Does the fallback chain actually help?

Compares the default single engine against the full chain on the same queries.
If the chain does not raise the fill rate, the extra engines are cost without
benefit and the default should stay narrow.

In [ ]:
chain = ",".join(ENGINES)

for query in QUERIES:
    solo = len(results[(query, "duckduckgo")])
    try:
        chained = len(text_to_urls(query, max_results=MAX_RESULTS, backend=chain))
    except Exception as exc:
        chained = f"error ({type(exc).__name__})"
    print(f"{query[:46]:<48} duckduckgo={solo:<3} chain={chained}")
    time.sleep(COOLDOWN_S)

## 5. Record the finding

Copy the availability/overlap tables into
`governance/model_cards/t2url-retrieval.md` under *Evaluation*, with today's date.
Engine behaviour drifts — an undated measurement is not evidence.

If a default changes as a result, note it in the model card's *Change log* and
update the defaults table in `ai/README.md`.